# Create Final Prediction Results CSV

Run this after notebooks 02 and 03. It merges the latest per-part outputs.

It prefers the rebuilt notebook outputs:

- `/prediction_results_part_a.csv`
- `/prediction_results_part_b.csv`

and falls back to the older `/DATASET/part_a_predictions.csv` and `/DATASET/part_b_predictions.csv` names if needed.

The final merged file is written to `/DATASET/prediction_results.csv`.

In [1]:
from pathlib import Path
import pandas as pd


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'DATASET').exists():
        return cwd
    if (cwd.parent / 'DATASET').exists():
        return cwd.parent
    raise FileNotFoundError('Could not find project root containing DATASET.')


def resolve_prediction_path(project_root: Path, dataset_dir: Path, *names: str) -> Path:
    for base in [project_root, dataset_dir]:
        for name in names:
            path = base / name
            if path.exists():
                return path
    raise FileNotFoundError(f'Could not find any of: {names}')


PROJECT_ROOT = resolve_project_root()
DATASET_DIR = PROJECT_ROOT / 'DATASET'
final_path = DATASET_DIR / 'prediction_results.csv'

part_a_path = resolve_prediction_path(PROJECT_ROOT, DATASET_DIR, 'prediction_results_part_a.csv', 'part_a_predictions.csv')
part_b_path = resolve_prediction_path(PROJECT_ROOT, DATASET_DIR, 'prediction_results_part_b.csv', 'part_b_predictions.csv')

part_a = pd.read_csv(part_a_path)
part_b = pd.read_csv(part_b_path)
prediction_results = pd.concat([part_a, part_b], ignore_index=True)
prediction_results = prediction_results.sort_values(['part', 'image_id']).reset_index(drop=True)
prediction_results.to_csv(final_path, index=False)
print('part_a:', part_a_path.relative_to(PROJECT_ROOT).as_posix())
print('part_b:', part_b_path.relative_to(PROJECT_ROOT).as_posix())
print(final_path.relative_to(PROJECT_ROOT).as_posix())
prediction_results.head()

part_a: prediction_results_part_a.csv
part_b: DATASET/prediction_results_part_b.csv
DATASET/prediction_results.csv


,part,split,image_id,image_path,actual_count,density_count_pred,aux_density_count_pred,strategy_name,blend_alpha,raw_predicted_count,predicted_count,signed_error,absolute_error
0,A,test,IMG_1,DATASET/part_A/test_data/images/IMG_1.jpg,172.000000,180.510757,547.184631,thresholded2_fused,0.0,324.442505,313.112245,141.112245,141.112245
1,A,test,IMG_10,DATASET/part_A/test_data/images/IMG_10.jpg,501.999939,201.983307,713.414185,thresholded2_fused,0.0,459.918884,443.857486,-58.142453,58.142453
2,A,test,IMG_100,DATASET/part_A/test_data/images/IMG_100.jpg,389.000031,189.102402,573.826660,thresholded2_fused,0.0,379.257568,366.013044,-22.986987,22.986987
3,A,test,IMG_101,DATASET/part_A/test_data/images/IMG_101.jpg,210.999969,176.589523,573.538574,thresholded2_fused,0.0,404.274567,390.156392,179.156423,179.156423
4,A,test,IMG_102,DATASET/part_A/test_data/images/IMG_102.jpg,223.000000,163.959732,449.217102,thresholded2_fused,0.0,228.270645,220.298922,-2.701078,2.701078


In [2]:
prediction_results.groupby('part')[['actual_count', 'predicted_count', 'signed_error', 'absolute_error']].agg(['mean', 'median', 'min', 'max']).round(3)

actual_count                      predicted_count                    \
             mean median   min     max            mean   median      min   
part                                                                       
A         432.945  307.5  66.0  2256.0         386.749  323.358  133.446   
B         123.703   93.0   9.0   539.0         111.333   87.502   24.369   

               signed_error                             absolute_error  \
           max         mean  median       min       max           mean   
part                                                                     
A     1479.015      -46.196  10.663 -1536.000  1172.015        189.047   
B      518.556      -12.369  -3.636  -328.997   107.556         30.066   

                                
       median    min       max  
part                            
A     106.285  2.701  1536.000  
B      16.662  0.131   328.997